In [ ]:
# /****************************************************************************************************************************
#  * Atharv Ravindra Sonawane
#  * BE-B - 46
#  *
#  * Problem statement : Write a Program for Matrix Multiplication using CUDA C.
#  *
#  * **************************************************************************************************************************/


%%writefile mat_mult.cu
#include <iostream>
using namespace std;

__global__ void matrixMul(int *A, int *B, int *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int sum = 0;
        for (int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

int main()
{
    int N = 3;
    int size = N * N * sizeof(int);

    int h_A[] = {1,2,3,4,5,6,7,8,9};
    int h_B[] = {9,8,7,6,5,4,3,2,1};
    int h_C[N*N];

    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 blocksPerGrid((N + 15)/16, (N + 15)/16);

    matrixMul<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cout << "Result Matrix:\n";
    for(int i = 0; i < N; i++)
    {
        for(int j = 0; j < N; j++)
        {
            cout << h_C[i*N + j] << " ";
        }
        cout << endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing mat_mult.cu


In [ ]:
!nvcc mat_mult.cu -o output

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./output

Result Matrix:
30 24 18 
84 69 54 
138 114 90 
